In [71]:
import json
import logging
import pandas as pd
import numpy as np
import pickle
import time
import os
import uuid
import getpass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
import pubchempy as pcp
from tqdm.notebook import tqdm
import sys
import duckdb

sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.pubchem_retrieval as pcr

creator_name = getpass.getuser()
timestamp = datetime.now().isoformat()

In [72]:
# Required: Input file with experimental data
COMPOUNDS_INPUT = "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"
#COMPOUNDS_INPUT = "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv"
#COMPOUNDS_INPUT = "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv"
#COMPOUNDS_INPUT = "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv"
#COMPOUNDS_INPUT = "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"

# Required: Database file path
DATABASE_PATH = "/Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb"

# Optional: Cache settings
PUBCHEM_CACHE_PATH = "/Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/pubchem_cache/pubchem_global_cache.pkl"  # Single global cache file
FORCE_CACHE_UPDATE = False  # If True, forces PubChem query even if data exists in cache

# Optional: Duplicate compound handling
ADD_COMPOUND_DUPLICATES = False  # If True, allows adding compounds that already exist in database

## Load and Validate Input Data

In [73]:
# Load and validate input data
print(f"Loading input data from: {COMPOUNDS_INPUT}")
delimiter = '\t' if COMPOUNDS_INPUT.endswith(('.tsv', '.tab', '.txt')) else ','
compounds_df = pd.read_csv(COMPOUNDS_INPUT, sep=delimiter)
print(f"Loaded {len(compounds_df)} rows from input table")

# Check for required columns (only core compound identification needed)
required_columns = ['inchi_key', 'label']
missing_columns = [col for col in required_columns if col not in compounds_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Get unique InChI keys for PubChem lookup
unique_inchi_keys = compounds_df['inchi_key'].dropna().unique()
print(f"\nUnique InChI keys to process: {len(unique_inchi_keys)}")

Loading input data from: /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv
Loaded 20 rows from input table

Unique InChI keys to process: 20


## PubChem Data Retrieval Functions

## Retrieve PubChem Data for All Compounds

In [74]:
# Load existing global cache
pubchem_cache = pcr.load_or_create_pubchem_cache(PUBCHEM_CACHE_PATH)

# Determine which compounds need PubChem lookup
if FORCE_CACHE_UPDATE:
    # Force update: query all compounds regardless of cache status
    compounds_to_fetch = list(unique_inchi_keys)
    compounds_in_cache = []
    print(f"Force update enabled - will query all {len(compounds_to_fetch)} compounds")
else:
    # Normal mode: only query compounds not in cache
    compounds_to_fetch = [key for key in unique_inchi_keys if key not in pubchem_cache]
    compounds_in_cache = [key for key in unique_inchi_keys if key in pubchem_cache]
    print(f"Compounds already in cache: {len(compounds_in_cache)}")
    print(f"Compounds needing PubChem lookup: {len(compounds_to_fetch)}")

if compounds_to_fetch:
    print(f"\nFetching PubChem data for {len(compounds_to_fetch)} compounds...")
    if FORCE_CACHE_UPDATE:
        print("Force update mode: updating existing cache entries")
    print("This may take several minutes depending on the number of compounds.")
    
    # Track how many were actually updated
    new_entries = 0
    updated_entries = 0
    
    # Fetch data for compounds
    for inchi_key in tqdm(compounds_to_fetch, desc="Fetching PubChem data"):
        
        was_in_cache = inchi_key in pubchem_cache
        
        # Get PubChem data
        pubchem_data = pcr.get_pubchem_data(inchi_key, timestamp)
        
        if pubchem_data:
            # Add retrieval timestamp to track when data was last updated
            pubchem_data["last_updated"] = timestamp
            pubchem_cache[inchi_key] = pubchem_data
            
            if was_in_cache:
                updated_entries += 1
            else:
                new_entries += 1
        else:
            print(f"No PubChem data found for {inchi_key}")
            # Store empty entry to avoid re-querying (unless force update is used)
            empty_entry = {
                "inchi_key": inchi_key,
                "pubchem_cid": "",
                "iupac_name": "",
                "synonyms": ["Undefined"],
                "error": "not_found_in_pubchem",
                "last_updated": timestamp
            }
            
            # Only store empty entry if not forcing updates or if it's truly new
            if not was_in_cache:
                pubchem_cache[inchi_key] = empty_entry
                new_entries += 1
        
        # Be respectful to PubChem API
        time.sleep(0.25)
    
    # Save updated cache
    pcr.save_pubchem_cache(pubchem_cache, PUBCHEM_CACHE_PATH)
    
    # Report what was done
    if FORCE_CACHE_UPDATE:
        print(f"Force update completed: {updated_entries} entries updated, {new_entries} entries added")
    else:
        print(f"Cache update completed: {new_entries} new entries added")
    
else:
    print("All compounds already in cache!")

print(f"\nPubChem data retrieval complete!")
print(f"Total compounds in global cache: {len(pubchem_cache)}")

# Show examples
successful_retrievals = [k for k, v in pubchem_cache.items() 
                        if v.get('pubchem_cid') and v.get('pubchem_cid') != '']
failed_retrievals = [k for k, v in pubchem_cache.items() 
                    if not v.get('pubchem_cid') or v.get('pubchem_cid') == '']

print(f"Successful PubChem retrievals in cache: {len(successful_retrievals)}")
print(f"Failed retrievals in cache: {len(failed_retrievals)}")

# Show cache age info if available
entries_with_dates = [v for v in pubchem_cache.values() if v.get('last_updated')]
if entries_with_dates:
    print(f"Cache entries with timestamps: {len(entries_with_dates)}")

if failed_retrievals:
    print(f"\nSome compounds not found in PubChem: {failed_retrievals[:5]}...")
    print("These will be created with minimal information from the input table.")

Loaded global PubChem cache with 482 entries from /Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/pubchem_cache/pubchem_global_cache.pkl
Compounds already in cache: 20
Compounds needing PubChem lookup: 0
All compounds already in cache!

PubChem data retrieval complete!
Total compounds in global cache: 482
Successful PubChem retrievals in cache: 478
Failed retrievals in cache: 4
Cache entries with timestamps: 482

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


## Add Compounds to Database

In [75]:
# Prepare compounds for database creation
print(f"Preparing {len(unique_inchi_keys)} compounds for database creation...")

compounds = {}
for idx, row in compounds_df.iterrows():
    inchi_key = row.get('inchi_key')
    if pd.notna(inchi_key):
        compounds[inchi_key] = row.to_dict()
        
# Create DuckDB database
print(f"\nOpening DuckDB database...")

# Add compounds to database (no atlas creation)
compounds_created, references_created, compounds_skipped, references_skipped = dbi.add_compounds_to_db(
    compounds_df, 
    pubchem_cache, 
    DATABASE_PATH, 
    creator_name,
    COMPOUNDS_INPUT,
    ADD_COMPOUND_DUPLICATES
)

print(f"\nDatabase update complete!")
print(f"   Database file: {DATABASE_PATH}")
print(f"   Created {compounds_created} new compounds")
print(f"   Skipped {compounds_skipped} duplicate compounds")
print(f"   Created {references_created} new RT/MZ references")
print(f"   Skipped {references_skipped} duplicate RT/MZ references")

dbi.validate_database(DATABASE_PATH)

Preparing 20 compounds for database creation...

Opening DuckDB database...
Adding compounds to database: /Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb
Found 482 existing compounds in database


Processing compounds: 100%|██████████| 20/20 [00:00<00:00, 2117.75it/s]


✓ Compounds added successfully!
   Total compounds in database: 482
   New compounds created: 0
   Compounds skipped (already existed): 20
   Total RT/MZ references in database: 872
   New RT/MZ references created: 0
   RT/MZ references skipped (duplicates): 20

Database update complete!
   Database file: /Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb
   Created 0 new compounds
   Skipped 20 duplicate compounds
   Created 0 new RT/MZ references
   Skipped 20 duplicate RT/MZ references

Database Validation:
   Compounds: 482
   RT/MZ References: 872
   Atlases: 0
   Atlas-Compound associations: 0
   Method combinations:
      HILICZ/negative: 466 references
      HILICZ/positive: 406 references

   No atlases found
